## 🛠️ Initialize

In [355]:
import pandas as pd
import numpy as np
import os
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

In [356]:
#FUnctions
def compute_fire_percentages(row):

    total = row["SURF_TOTAL"]

    # avoid division by zero
    if total == 0 or pd.isna(total):
        return {
            "FORET_%": 0,
            "MAQUIS_%": 0,
            "BROUSSAILLES_%": 0
        }

    foret_pct = (row["TOT_FORET"] / total) * 100
    maquis_pct = (row["TOT_MAQUIS"] / total) * 100
    broussaille_pct = (row["TOT_BROUSSAILLES"] / total) * 100

    return {
        "FORET_%": round(foret_pct, 2),
        "MAQUIS_%": round(maquis_pct, 2),
        "BROUSSAILLES_%": round(broussaille_pct, 2)
        }

def get_season(month):
    if month in [3, 4, 5]:
        return 'SPRING'
    elif month in [6, 7, 8]:
        return 'SUMMER'
    elif month in [9, 10, 11]:
        return 'AUTUMN'
    else:
        return 'WINTER'

In [357]:
# Charger le fichier
df1 = pd.read_excel("clean_dataset.xlsx")
print(df1.columns)
df1.head(5)

Index(['DAIRA', 'COMMUNE', 'FORET', 'LONGITUDE', 'LATITUDE', 'DATE_DECL',
       'HEURE_DECL', 'DATE_INT', 'HEURE_INT', 'DATE_EXT', 'HEURE_EXT',
       'ESSENCE', 'TOT_FORET', 'TOT_MAQUIS', 'TOT_BROUSSAILLES', 'SURF_TOTAL',
       'CAUSE', 'SIGNALE', 'ORGANISMES', 'DEGATS', 'DECL_YEAR', 'DECL_MONTH',
       'DECL_DAY', 'INT_YEAR', 'INT_MONTH', 'INT_DAY', 'EXT_YEAR', 'EXT_MONTH',
       'EXT_DAY', 'DECL_HOUR', 'INT_HOUR', 'EXT_HOUR', 'METEO_TEMP',
       'METEO_VENTE_VITESSE', 'METEO_VENTE_DIRECTION', 'LOCATION_NAME'],
      dtype='object')


,DAIRA,COMMUNE,FORET,LONGITUDE,LATITUDE,DATE_DECL,HEURE_DECL,DATE_INT,HEURE_INT,DATE_EXT,...,EXT_YEAR,EXT_MONTH,EXT_DAY,DECL_HOUR,INT_HOUR,EXT_HOUR,METEO_TEMP,METEO_VENTE_VITESSE,METEO_VENTE_DIRECTION,LOCATION_NAME
0,HAMMAM DEBAGH,ROKNIA,DJ DEBAGH,NaN,NaN,2010-07-02,1900-01-01 10:20:00,2010-07-02,1900-01-01 10:20:00,2010-07-02,...,2010,7,2,10.0,10,15.0,26.07,5.40,S+NE,NaN
1,GUELAAT BOU SBAA,BENI MEZLINE,KEF AMAR,NaN,NaN,2010-07-09,1900-01-01 11:10:00,2010-07-09,1900-01-01 11:10:00,2010-07-09,...,2010,7,9,11.0,11,15.0,26.50,6.01,NE+S,NaN
2,HAMMAM N'BAIL,OUED CHEHAM,SAFIET AOUAYED,NaN,NaN,2010-07-12,1900-01-01 16:25:00,2010-07-12,1900-01-01 16:25:00,2010-07-12,...,2010,7,12,16.0,16,17.0,28.60,6.06,S+N,NaN
3,HAMMAM N'BAIL,OUED CHEHAM,OUED GHANEM,NaN,NaN,2010-07-13,1900-01-01 11:45:00,2010-07-13,1900-01-01 11:45:00,2010-07-13,...,2010,7,13,11.0,11,15.0,28.27,5.53,N,NaN
4,HELIOPOLIS,EL FEDJOUDJ,AIN TOUTA,NaN,NaN,2010-07-14,1900-01-01 12:30:00,2010-07-14,1900-01-01 12:30:00,2010-07-14,...,2010,7,14,12.0,12,15.0,30.26,6.85,S+N,NaN


## 💥THEME 2 - FIRE DAMAGE PREDICTION

### 👩🏻‍🍳 Prepare Data

In [358]:
df1.columns

Index(['DAIRA', 'COMMUNE', 'FORET', 'LONGITUDE', 'LATITUDE', 'DATE_DECL',
       'HEURE_DECL', 'DATE_INT', 'HEURE_INT', 'DATE_EXT', 'HEURE_EXT',
       'ESSENCE', 'TOT_FORET', 'TOT_MAQUIS', 'TOT_BROUSSAILLES', 'SURF_TOTAL',
       'CAUSE', 'SIGNALE', 'ORGANISMES', 'DEGATS', 'DECL_YEAR', 'DECL_MONTH',
       'DECL_DAY', 'INT_YEAR', 'INT_MONTH', 'INT_DAY', 'EXT_YEAR', 'EXT_MONTH',
       'EXT_DAY', 'DECL_HOUR', 'INT_HOUR', 'EXT_HOUR', 'METEO_TEMP',
       'METEO_VENTE_VITESSE', 'METEO_VENTE_DIRECTION', 'LOCATION_NAME'],
      dtype='object')

In [359]:
drop_cols = [ 'LONGITUDE', 'LATITUDE', 'DATE_INT', 'HEURE_INT', 'INT_YEAR', 'INT_MONTH', 'INT_DAY',
             'INT_HOUR', 'LOCATION_NAME', 'EXT_YEAR', 'EXT_MONTH', 'TOT_BROUSSAILLES', 'TOT_MAQUIS',
             'TOT_FORET', 'SURF_TOTAL'
             ]
#drop anythin that starts with HEURE or ends with _DAY
df2 = df1.copy()
df2 = df2.drop(columns=drop_cols)
df2['DEGATS'] = df2['DEGATS'].fillna(df2['DEGATS'].mean())
print(df2.columns)
df2.head(5)

Index(['DAIRA', 'COMMUNE', 'FORET', 'DATE_DECL', 'HEURE_DECL', 'DATE_EXT',
       'HEURE_EXT', 'ESSENCE', 'CAUSE', 'SIGNALE', 'ORGANISMES', 'DEGATS',
       'DECL_YEAR', 'DECL_MONTH', 'DECL_DAY', 'EXT_DAY', 'DECL_HOUR',
       'EXT_HOUR', 'METEO_TEMP', 'METEO_VENTE_VITESSE',
       'METEO_VENTE_DIRECTION'],
      dtype='object')


,DAIRA,COMMUNE,FORET,DATE_DECL,HEURE_DECL,DATE_EXT,HEURE_EXT,ESSENCE,CAUSE,SIGNALE,...,DEGATS,DECL_YEAR,DECL_MONTH,DECL_DAY,EXT_DAY,DECL_HOUR,EXT_HOUR,METEO_TEMP,METEO_VENTE_VITESSE,METEO_VENTE_DIRECTION
0,HAMMAM DEBAGH,ROKNIA,DJ DEBAGH,2010-07-02,1900-01-01 10:20:00,2010-07-02,1900-01-01 15:00:00,MAQ,INC,PV,...,400000.0,2010,7,2,2,10.0,15.0,26.07,5.40,S+NE
1,GUELAAT BOU SBAA,BENI MEZLINE,KEF AMAR,2010-07-09,1900-01-01 11:10:00,2010-07-09,1900-01-01 15:00:00,MAQ,INC,PV,...,200000.0,2010,7,9,9,11.0,15.0,26.50,6.01,NE+S
2,HAMMAM N'BAIL,OUED CHEHAM,SAFIET AOUAYED,2010-07-12,1900-01-01 16:25:00,2010-07-12,1900-01-01 17:05:00,BRS,INC,BM,...,50000.0,2010,7,12,12,16.0,17.0,28.60,6.06,S+N
3,HAMMAM N'BAIL,OUED CHEHAM,OUED GHANEM,2010-07-13,1900-01-01 11:45:00,2010-07-13,1900-01-01 15:25:00,BRS,INC,BM,...,250000.0,2010,7,13,13,11.0,15.0,28.27,5.53,N
4,HELIOPOLIS,EL FEDJOUDJ,AIN TOUTA,2010-07-14,1900-01-01 12:30:00,2010-07-14,1900-01-01 15:00:00,BRS,INC,PV,...,200000.0,2010,7,14,14,12.0,15.0,30.26,6.85,S+N


In [360]:
df3 = df2.copy()
df3['MONTH'] = df3['DECL_MONTH']
df3.drop(columns=['DECL_MONTH'], inplace=True)
df3['SEASON'] = df3['MONTH'].apply(get_season)


df3['DATE_DECL'] = pd.to_datetime(df3['DATE_DECL'], format='%Y-%m-%d', errors='coerce')
df3['DATE_EXT']  = pd.to_datetime(df3['DATE_EXT'], format='%Y-%m-%d', errors='coerce')

base = pd.to_datetime('1900-01-01')

decl_time = pd.to_datetime(df3['HEURE_DECL'], errors='coerce') - base
ext_time  = pd.to_datetime(df3['HEURE_EXT'], errors='coerce') - base

# rebuild full datetime correctly
df3['DECL'] = df3['DATE_DECL'] + decl_time
df3['EXT']  = df3['DATE_EXT'] + ext_time


# duration in minutes
df3['DUREE'] = (df3['EXT'] - df3['DECL']).dt.total_seconds() / 60

df3.drop(columns=['HEURE_DECL', 'DATE_EXT', 'HEURE_EXT', 'DECL', 'EXT'], inplace=True)



#BINARY ENCODING
df3['CAUSE'] = df3['CAUSE'].map({'INC': 1, 'CON': 0})

df3

,DAIRA,COMMUNE,FORET,DATE_DECL,ESSENCE,CAUSE,SIGNALE,ORGANISMES,DEGATS,DECL_YEAR,DECL_DAY,EXT_DAY,DECL_HOUR,EXT_HOUR,METEO_TEMP,METEO_VENTE_VITESSE,METEO_VENTE_DIRECTION,MONTH,SEASON,DUREE
0,HAMMAM DEBAGH,ROKNIA,DJ DEBAGH,2010-07-02,MAQ,1,PV,SF+PC+DW,4.000000e+05,2010,2,2,10.0,15.0,26.07,5.40,S+NE,7,SUMMER,280.0
1,GUELAAT BOU SBAA,BENI MEZLINE,KEF AMAR,2010-07-09,MAQ,1,PV,SF+PC+DW,2.000000e+05,2010,9,9,11.0,15.0,26.50,6.01,NE+S,7,SUMMER,230.0
2,HAMMAM N'BAIL,OUED CHEHAM,SAFIET AOUAYED,2010-07-12,BRS,1,BM,SF,5.000000e+04,2010,12,12,16.0,17.0,28.60,6.06,S+N,7,SUMMER,40.0
3,HAMMAM N'BAIL,OUED CHEHAM,OUED GHANEM,2010-07-13,BRS,1,BM,SF,2.500000e+05,2010,13,13,11.0,15.0,28.27,5.53,N,7,SUMMER,220.0
4,HELIOPOLIS,EL FEDJOUDJ,AIN TOUTA,2010-07-14,BRS,1,PV,SF,2.000000e+05,2010,14,14,12.0,15.0,30.26,6.85,S+N,7,SUMMER,150.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
538,HAMMAM N'BAIL,HAMMAM N'BAILS,SOUIBINA,2025-08-27,MAQ,1,PV,SF+PC+DW+EAPC+ANP+EMPS+VOL,1.131180e+07,2025,27,28,19.0,2.0,33.95,7.32,SO,8,SUMMER,365.0
539,BOUCHEGOUF,MEDJEZ SFA,BENI SALAH,2025-08-31,MAQ+BRS,1,PV,SF+PC+DW+EAPC+SN+ANP+EMPS+VOL,1.131180e+07,2025,31,2,14.0,11.0,27.65,4.84,S,8,SUMMER,2735.0
540,HAMMAM N'BAIL,DAHOUARA,DJ AOURES,2025-09-03,MAQ,1,PV,SF+PC+EAPC+EMPS+DW,1.131180e+07,2025,3,3,15.0,17.0,24.74,4.87,E+NE,9,AUTUMN,162.0
541,HELIOPOLIS,HELIOPOLIS,AIN RIHANA,2025-09-07,MAQ,1,PV,SF+PC+EAPC+DW+EMPS,1.000000e+05,2025,7,7,14.0,16.0,30.95,8.38,SO+NO,9,AUTUMN,130.0


### ✂️ Split Tain/Test

In [361]:
folder = "Cleaned Data/Theme2"
df4= df3.copy()
#  make sure df is sorted by date first
df4 = df4.sort_values("DATE_DECL").reset_index(drop=True)

#save full cleaned dataset
df4.to_excel(f"{folder}/clean_dataset.xlsx", index=False)

# compute split index (80% train, 20% test)
split_idx = int(len(df4) * 0.8)

# split
df_train = df4.iloc[:split_idx].copy()
df_test = df4.iloc[split_idx:].copy()

# (optional) sanity check
print("Total:", len(df4))
print("Train:", len(df_train))
print("Test:", len(df_test))

# save to Excel
# folder path

os.makedirs(folder, exist_ok=True)

# save files
df_train.to_excel(f"{folder}/split_train.xlsx", index=False)
df_test.to_excel(f"{folder}/split_test.xlsx", index=False)

print("Saved train and test files in:", folder)

Total: 543
Train: 434
Test: 109
Saved train and test files in: Cleaned Data/Theme2


In [362]:
#for train
df5 = pd.read_excel("Cleaned Data/Theme2/split_train.xlsx")

In [363]:
kpgkpkp = 

SyntaxError: invalid syntax (3215493002.py, line 1)

In [379]:
#for test
df5 = pd.read_excel("Cleaned Data/Theme2/split_test.xlsx")

### 🧊 Freeze Values (train)

In [364]:
# 1. fire count per DAIRA
daira_counts = df5['DAIRA'].value_counts().reset_index()
daira_counts.columns = ['DAIRA', 'FIRE_COUNT']

# 2. define groups (TRAIN-based logic)
strong_dairas = daira_counts[daira_counts['FIRE_COUNT'] >= 80]['DAIRA'].tolist()

medium_dairas = daira_counts[
    (daira_counts['FIRE_COUNT'] < 80) &
    (daira_counts['FIRE_COUNT'] >= 20)
]['DAIRA'].tolist()

weak_dairas = daira_counts[daira_counts['FIRE_COUNT'] < 20]['DAIRA'].tolist()

In [365]:
commune_counts = df5['COMMUNE'].value_counts().reset_index()
commune_counts.columns = ['COMMUNE', 'FIRE_COUNT']

strong_communes = commune_counts[commune_counts['FIRE_COUNT'] >= 20]['COMMUNE'].tolist()

medium_communes = commune_counts[
    (commune_counts['FIRE_COUNT'] < 20) &
    (commune_counts['FIRE_COUNT'] >= 4)
]['COMMUNE'].tolist()

weak_communes = commune_counts[commune_counts['FIRE_COUNT'] < 4]['COMMUNE'].tolist()

In [366]:
forest_counts = df5['FORET'].value_counts().reset_index()
forest_counts.columns = ['FORET', 'FIRE_COUNT']

strong_forests = forest_counts[forest_counts['FIRE_COUNT'] >= 6]['FORET'].tolist()

medium_forests = forest_counts[
    (forest_counts['FIRE_COUNT'] < 6) &
    (forest_counts['FIRE_COUNT'] >= 3)
]['FORET'].tolist()

weak_forests = forest_counts[forest_counts['FIRE_COUNT'] < 3]['FORET'].tolist()

In [367]:
const_wind_directions = ['N', 'NE', 'E', 'SE', 'S', 'SO', 'O', 'NO']
const_essences = [
 'BRS',
 'CL',
 'CV',
 'CYP',
 'CZ',
 'EUC',
 'ESSENCE',
 'FORETS',
 'GUENDOUL',
 'MAQ',
 'PA',
 'REB',
 'TPARCOUR'
]
const_seasons = ['SUMMER', 'AUTUMN', 'WINTER', 'SPRING']
const_signals = df5['SIGNALE'].unique().tolist()
const_organismes = sorted(
    set(
        org.strip()
        for row in df5['ORGANISMES'].dropna()
        for org in row.split('+')
    )
)

### 👷 Feature Engennering

In [380]:
df6 = df5.copy()

# =========================
# 1. DAIRA
# =========================

df6['DAIRA_FIRE_COUNT'] = df6['DAIRA'].map(daira_counts.set_index('DAIRA')['FIRE_COUNT'])

def map_daira(x):
    if x in strong_dairas:
        return x
    else:
        return 'OTHER'

df6['DAIRA_GROUP'] = df6['DAIRA'].apply(
    lambda x: 'STRONG' if x in strong_dairas else
              'MEDIUM' if x in medium_dairas else
              'WEAK'   if x in weak_dairas else 'OTHER'
)

df6['DAIRA_2'] = df6['DAIRA'].apply(map_daira)


# =========================
# 2. COMMUNE
# =========================

df6['COMMUNE_FIRE_COUNT'] = df6['COMMUNE'].map(commune_counts.set_index('COMMUNE')['FIRE_COUNT'])

def map_commune(x):
    if x in strong_communes:
        return x
    else:
        return 'OTHER'

df6['COMMUNE_GROUP'] = df6['COMMUNE'].apply(
    lambda x: 'STRONG' if x in strong_communes else
              'MEDIUM' if x in medium_communes else
              'WEAK'   if x in weak_communes else 'OTHER'
)

df6['COMMUNE_2'] = df6['COMMUNE'].apply(map_commune)


# =========================
# 3. FORET
# =========================

df6['FORET_FIRE_COUNT'] = df6['FORET'].map(forest_counts.set_index('FORET')['FIRE_COUNT'])

def map_forest(x):
    if x in strong_forests:
        return x
    else:
        return 'OTHER'

df6['FORET_GROUP'] = df6['FORET'].apply(
    lambda x: 'STRONG' if x in strong_forests else
              'MEDIUM' if x in medium_forests else
              'WEAK'   if x in weak_forests else 'OTHER'
)

df6['FORET_2'] = df6['FORET'].apply(map_forest)
df6.head()

,DAIRA,COMMUNE,FORET,DATE_DECL,ESSENCE,CAUSE,SIGNALE,ORGANISMES,DEGATS,DECL_YEAR,...,DUREE,DAIRA_FIRE_COUNT,DAIRA_GROUP,DAIRA_2,COMMUNE_FIRE_COUNT,COMMUNE_GROUP,COMMUNE_2,FORET_FIRE_COUNT,FORET_GROUP,FORET_2
0,KHEZARA,KHEZARAS,METOUIA,2020-09-18,BRS,1,PV,SF+PC,1000000.0,2020,...,90.0,14.0,WEAK,OTHER,3.0,WEAK,OTHER,1.0,WEAK,OTHER
1,OUED ZENATI,BORDJ SABAT,SILA,2020-09-18,BRS,0,BM,SF+PC,1000000.0,2020,...,915.0,27.0,MEDIUM,OTHER,22.0,STRONG,BORDJ SABAT,NaN,OTHER,OTHER
2,OUED ZENATI,BORDJ SABAT,EL SATHA,2020-09-18,MAQ,1,BM,SF+PC,1000000.0,2020,...,135.0,27.0,MEDIUM,OTHER,22.0,STRONG,BORDJ SABAT,NaN,OTHER,OTHER
3,AIN HESSAINIA,SELLAOUA ANNOUNA,DJ LEZREG,2020-09-19,BRS,1,PV,SF+PC,600000.0,2020,...,480.0,5.0,WEAK,OTHER,1.0,WEAK,OTHER,1.0,WEAK,OTHER
4,AIN HESSAINIA,SELLAOUA ANNOUNA,DJ LEZREG,2020-09-25,BRS,1,PV,SF+PC,600000.0,2020,...,174.0,5.0,WEAK,OTHER,1.0,WEAK,OTHER,1.0,WEAK,OTHER


### 🧊 Freeze Forest Context (train)

In [372]:
forest_context = (
    df6.groupby("FORET")["ESSENCE"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
    .reset_index()
    .rename(columns={"ESSENCE": "ESSENCE"})
)

### 🧊 Append Forest Context (test)

In [381]:
existing_forests = set(forest_context["FORET"])

missing_df = df6[~df6["FORET"].isin(existing_forests)]

def safe_mode(x):
    m = x.mode()
    return m.iloc[0] if not m.empty else x.dropna().iloc[0]

new_context = (
    missing_df.groupby("FORET")["ESSENCE"]
    .apply(safe_mode)
    .reset_index(name="ESSENCE")
)

new_context = new_context.rename(columns={"ESSENCE": "ESSENCE"})

forest_context = pd.concat([forest_context, new_context], ignore_index=True)

forest_context = forest_context.drop_duplicates(subset=["FORET"], keep="first").reset_index(drop=True)

In [382]:
forest_context

,FORET,ESSENCE
0,AIN AMAR,MAQ+BRS
1,AIN BEN ASSOUL,MAQ
2,AIN BEN KHERFENE,BRS
3,AIN BOUGUERRA,CL+MAQ
4,AIN CHEKA,PA
...,...,...
244,SERDOUNE,MAQ
245,SILA,BRS
246,SOUIBINA,MAQ
247,SÉRIE SIDI SELAM,MAQ


### 🌲 Apply Forrest Context

In [383]:
df6 = df6.merge(forest_context, on="FORET", how="left", suffixes=("", "_ctx"))

for col in forest_context.columns:
    if col != "FORET" and col in df6.columns:
        df6[col] = df6[col + "_ctx"]
        df6.drop(columns=[col + "_ctx"], inplace=True)


## 👨🏻‍💻 Prepare for Model

In [384]:
df7 = df6.copy()
df7['COMMUNE'] = df6['COMMUNE_2']
df7['FORET'] = df6['FORET_2']
df7['DAIRA'] = df6['DAIRA_2']
df7['MONTH_SIN'] = np.sin(2 * np.pi * df6['MONTH'] / 12)
df7['MONTH_COS'] = np.cos(2 * np.pi * df6['MONTH'] / 12)
df7 = df7.drop(columns=['COMMUNE_2', 'FORET_2', 'DAIRA_2', 'MONTH', 'DATE_DECL'])

In [385]:
def encode_multilabel_fixed(df, col, values, prefix):

    split_series = df[col].fillna("").apply(lambda x: x.split("+"))

    for val in values:
        df[f"{prefix}_{val}"] = split_series.apply(lambda x: int(val in x))

    return df.drop(columns=[col])


def encode_onehot_fixed(df, col, values, prefix):

    for val in values:
        df[f"{prefix}_{val}"] = (df[col] == val).astype(int)

    return df.drop(columns=[col])


df8 = df7.copy()

# --- multi-label features ---
df8 = encode_multilabel_fixed(df8, "ESSENCE", const_essences, "ESSENCE")
df8 = encode_multilabel_fixed(df8, "METEO_VENTE_DIRECTION", const_wind_directions, "WIND")
df8 = encode_multilabel_fixed(df8, "ORGANISMES", const_organismes, "ORGANISMES")

# --- single-label categorical ---
cat_cols = ['DAIRA', 'COMMUNE', 'FORET', 'DAIRA_GROUP', 'FORET_GROUP', 'COMMUNE_GROUP', 'SIGNALE', 'SEASON']

df8 = pd.get_dummies(df8, columns=cat_cols, drop_first=False)

df8 = df8.loc[:, ~df8.columns.str.endswith("_OTHER")]



#Test droping location columns
drop_cols = [
    col for col in df8.columns
    if (
        (
            #col.startswith("FORET_")
            #or col.startswith("COMMUNE_")
            # or col.startswith("DAIRA_")
        )
        and ("_FIRE_COUNT" not in col)
        and ("_PERCENT" not in col)
        #and ("_GROUP" not in col)
    )
]

#test removing essence
#drop_cols += [col for col in df8.columns if col.startswith("ESSENCE_")]

df8 = df8.drop(columns=drop_cols)

print("Final shape:", df8.shape)

Final shape: (109, 81)


In [378]:
#for train
df8.to_excel("Cleaned Data/Theme2/train.xlsx", index=False)

In [386]:
#for test
df8.to_excel("Cleaned Data/Theme2/test.xlsx", index=False)

## 🤖 AI Model

In [420]:
df_train = pd.read_excel("Cleaned Data/Theme2/train.xlsx")
X_train = df_train.drop(columns=['DEGATS'])
y_train = df_train['DEGATS']
y_train = np.log1p(y_train)

In [421]:
df_test = pd.read_excel("Cleaned Data/Theme2/test.xlsx")
X_test = df_test.drop(columns=['DEGATS'])
y_test = df_test['DEGATS']
y_test = np.log1p(y_test)

# ALIGN
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
X_test.columns

Index(['CAUSE', 'DECL_YEAR', 'DECL_DAY', 'EXT_DAY', 'DECL_HOUR', 'EXT_HOUR',
       'METEO_TEMP', 'METEO_VENTE_VITESSE', 'DUREE', 'DAIRA_FIRE_COUNT',
       'COMMUNE_FIRE_COUNT', 'FORET_FIRE_COUNT', 'MONTH_SIN', 'MONTH_COS',
       'ESSENCE_BRS', 'ESSENCE_CL', 'ESSENCE_CV', 'ESSENCE_CYP', 'ESSENCE_CZ',
       'ESSENCE_EUC', 'ESSENCE_ESSENCE', 'ESSENCE_FORETS', 'ESSENCE_GUENDOUL',
       'ESSENCE_MAQ', 'ESSENCE_PA', 'ESSENCE_REB', 'ESSENCE_TPARCOUR',
       'WIND_N', 'WIND_NE', 'WIND_E', 'WIND_SE', 'WIND_S', 'WIND_SO', 'WIND_O',
       'WIND_NO', 'ORGANISMES_AEPC', 'ORGANISMES_ANP', 'ORGANISMES_CIT',
       'ORGANISMES_COMR', 'ORGANISMES_DW', 'ORGANISMES_EAPC',
       'ORGANISMES_EMPS', 'ORGANISMES_PC', 'ORGANISMES_POP', 'ORGANISMES_SF',
       'ORGANISMES_SN', 'ORGANISMES_VLO', 'ORGANISMES_VOL', 'DAIRA_BOUCHEGOUF',
       'DAIRA_HAMMAM DEBAGH', 'COMMUNE_BORDJ SABAT', 'COMMUNE_BOUATI MAHMOUD',
       'COMMUNE_BOUCHEGOUF', 'COMMUNE_BOUHAMDANE', 'COMMUNE_HAMMAM N'BAILS',
       'COMMUNE_M

In [422]:
# test removing some features
cols_to_drop = [col for col in X_train.columns if col.startswith("FORET_") or col.startswith("COMMUNE_") or col.startswith("DAIRA_")]
cols_to_drop = cols_to_drop + [col for col in X_train.columns if col.startswith("SIGNALE_") or col.startswith("ORGANISMES_") 
                               or col.startswith("ESSENCE_")]

X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

In [423]:
X_test.columns

Index(['CAUSE', 'DECL_YEAR', 'DECL_DAY', 'EXT_DAY', 'DECL_HOUR', 'EXT_HOUR',
       'METEO_TEMP', 'METEO_VENTE_VITESSE', 'DUREE', 'MONTH_SIN', 'MONTH_COS',
       'WIND_N', 'WIND_NE', 'WIND_E', 'WIND_SE', 'WIND_S', 'WIND_SO', 'WIND_O',
       'WIND_NO', 'SEASON_AUTUMN', 'SEASON_SUMMER'],
      dtype='object')

In [410]:
model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='rmse',
    random_state=42
)


model.fit(X_train, y_train)

y_pred_log = model.predict(X_test)

y_pred = np.expm1(y_pred_log)


y_pred = np.clip(y_pred, 0, None)


mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

MAE  : 21900597.3901
RMSE : 136203016.8650
R2   : 0.0087


In [424]:
#fill all nan with mean
X_test = X_test.fillna(X_test.mean())

In [425]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# -----------------------
# MODEL
# -----------------------
model = LinearRegression()

# y_train is already log → OK
model.fit(X_train, y_train)

# -----------------------
# PREDICTION (still log space)
# -----------------------
y_pred_log = model.predict(X_test)

# back to original scale
y_pred = np.expm1(y_pred_log)
y_pred = np.clip(y_pred, 0, None)

# IMPORTANT: also transform y_test
y_test_original = np.expm1(y_test)

# -----------------------
# METRICS (ALL in original space)
# -----------------------
mae = mean_absolute_error(y_test_original, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_original, y_pred))
r2 = r2_score(y_test_original, y_pred)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

MAE  : 20168043388770712322515664896.0000
RMSE : 210560554658656960677851168768.0000
R2   : -2369129108384482983650922152146320991715328.0000
